In [ ]:
import re
import warnings
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import joblib

from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = Path("openf1_data")
MODEL_FILE = "multitask_strategy_model_combined.keras"
PREPROC_FILE = "multitask_preprocessing_combined.joblib"

RACE_CONTEXT_FILE = "race_context_input.csv"
FORECAST_FILE = "forecast_weather_detailed_rainy.csv"

WEATHER_FEATURES = ["air_temperature", "track_temperature", "humidity", "rainfall", "wind_speed"]
VALID_COMPOUNDS = ["SOFT", "MEDIUM", "HARD", "INTERMEDIATE", "WET"]
DRY_COMPOUNDS = ["SOFT", "MEDIUM", "HARD"]
DEFAULT_PIT_LOSS = 22.0

In [ ]:
def parse_session_key(path: Path):
    m = re.search(r"_session_(\d+)\.csv$", path.name)
    return int(m.group(1)) if m else None

def safe_read_csv(path: Path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

def format_time(sec):
    m = int(sec // 60)
    s = sec - m * 60
    return f"{m}:{s:06.3f}"

def load_session_maps():
    sessions = safe_read_csv(DATA_DIR / "sessions.csv")
    if sessions.empty:
        return {}, {}, {}

    for c in ["session_key", "meeting_key", "session_name", "location", "circuit_short_name"]:
        if c not in sessions.columns:
            sessions[c] = np.nan

    sessions["track_name"] = sessions["circuit_short_name"].fillna(sessions["location"]).fillna(sessions["meeting_key"].astype(str))
    return (
        dict(zip(sessions["session_key"], sessions["track_name"])),
        dict(zip(sessions["session_key"], sessions["meeting_key"])),
        dict(zip(sessions["session_key"], sessions["session_name"].astype(str))),
    )

session_to_track, _, _ = load_session_maps() if DATA_DIR.exists() else ({}, {}, {})

def encode_with_unknown(le, val):
    """Encode with fallback to first class if unknown"""
    val = str(val)
    if val in set(le.classes_):
        return int(le.transform([val])[0])
    return 0  # Fallback to first class

def estimate_track_pit_loss_map():
    """Estimate pit loss time per track from historical data"""
    out = {}

    if not DATA_DIR.exists():
        return out

    for pf in sorted(DATA_DIR.glob("pit_session_*.csv")):
        sid = parse_session_key(pf)
        if sid is None:
            continue
        track = str(session_to_track.get(sid, "UNKNOWN_TRACK"))
        p = safe_read_csv(pf)
        if p.empty:
            continue

        dcol = next((c for c in ["pit_duration", "duration", "pit_time"]
                    if c in p.columns), None)
        if dcol:
            d = pd.to_numeric(p[dcol], errors="coerce").dropna()
            if len(d):
                out[track] = float(np.median(d) + 15.0)
    return out

def forecast_to_sequence_and_summary(forecast_df, weather_features, max_timesteps, seq_mean, seq_std):
    """Convert forecast data to normalized sequence and summary stats"""
    f = forecast_df.copy()

    for c in weather_features:
        if c not in f.columns:
            f[c] = np.nan
        f[c] = pd.to_numeric(f[c], errors="coerce")

    f = f[weather_features].ffill().bfill().fillna(0.0)

    # Pad and normalize consistently with training
    seq = pad_sequences([f.values.astype("float32")],
                       maxlen=max_timesteps,
                       padding="post",
                       dtype="float32")
    seq = (seq - seq_mean) / seq_std

    summary = {
        "air_temp_mean": float(f["air_temperature"].mean()),
        "track_temp_mean": float(f["track_temperature"].mean()),
        "humidity_mean": float(f["humidity"].mean()),
        "rain_minutes_ratio": float((f["rainfall"] > 0).mean()),
        "wind_speed_mean": float(f["wind_speed"].mean()),
    }
    return seq, summary

In [ ]:
def split_laps(total_laps, n_stints):
    """Split total laps across stints as evenly as possible"""
    b = total_laps // n_stints
    r = total_laps % n_stints
    return [b + (1 if i < r else 0) for i in range(n_stints)]

def build_candidates(start_comp, max_stops=3, wet=False):
    """Generate all valid strategy candidates"""
    pool = VALID_COMPOUNDS if wet else DRY_COMPOUNDS
    start_comp = str(start_comp).upper()
    if start_comp not in pool:
        start_comp = "MEDIUM" if "MEDIUM" in pool else pool[0]

    cands = []
    for s in range(max_stops + 1):
        if s == 0:
            cands.append([start_comp])
        else:
            for seq in product(pool, repeat=s):
                full = [start_comp] + list(seq)
                # Skip strategies with no compound changes
                if len(set(full)) == 1:
                    continue
                cands.append(full)
    return cands

In [ ]:
def simulate_with_gru_enhanced(model, pre, race_context, seq_input, weather_summary, compounds, pit_loss):
    """
    Enhanced simulation using FastF1 tire degradation data

    Improvements over original:
    - Uses learned tire degradation rates
    - Better weather impact modeling
    - Track-specific adjustments
    """
    # Encode categorical features
    team_id = encode_with_unknown(pre["team_le"], race_context["team_name"])
    track_id = encode_with_unknown(pre["track_le"], race_context["track_name"])
    start_id = encode_with_unknown(pre["start_le"], race_context["starting_compound"])

    # Get average tire degradation from context (or use default)
    avg_tire_deg = race_context.get("avg_tire_degradation", 0.025)

    # Prepare numerical features (must match training normalization)
    num_vec = np.array([[
        race_context["starting_position"],
        weather_summary["air_temp_mean"],
        weather_summary["track_temp_mean"],
        weather_summary["humidity_mean"],
        weather_summary["rain_minutes_ratio"],
        weather_summary["wind_speed_mean"],
        avg_tire_deg  # NEW: tire degradation feature
    ]], dtype="float32")

    # Get model predictions
    preds = model.predict({
        "weather_seq": seq_input,
        "team_in": np.array([team_id], dtype=np.int32),
        "track_in": np.array([track_id], dtype=np.int32),
        "start_comp_in": np.array([start_id], dtype=np.int32),
        "num_in": num_vec,
    }, verbose=0)

    # Safely unpack predictions
    if isinstance(preds, dict):
        p_stops, p_tire, p_time = list(preds.values())
    else:
        p_stops, p_tire, p_time = preds

    # === ENHANCED PHYSICS-BASED LAP TIME SIMULATION ===
    total_laps = int(race_context["race_laps"])

    # Base lap time estimation (track + weather effects)
    base_lap = (92.0 +
                0.08 * max(weather_summary["track_temp_mean"] - 35.0, 0) +
                6.0 * weather_summary["rain_minutes_ratio"])

    # Enhanced compound performance deltas (calibrated from FastF1 data)
    comp_delta = {
        "SOFT": -0.30,      # Faster
        "MEDIUM": 0.0,      # Baseline
        "HARD": 0.22,       # Slower
        "INTERMEDIATE": 1.2 if weather_summary["rain_minutes_ratio"] < 0.5 else -0.5,  # Context-dependent
        "WET": 2.0 if weather_summary["rain_minutes_ratio"] < 0.3 else -1.0
    }

    # Enhanced degradation per lap (learned from FastF1 data, scaled by context)
    deg_multiplier = 1.0 + 0.5 * avg_tire_deg  # Scale by learned degradation
    degr = {
        "SOFT": 0.040 * deg_multiplier,
        "MEDIUM": 0.025 * deg_multiplier,
        "HARD": 0.018 * deg_multiplier,
        "INTERMEDIATE": 0.032 * deg_multiplier,
        "WET": 0.042 * deg_multiplier
    }

    # Simulate each stint
    stint_laps = split_laps(total_laps, len(compounds))
    detailed = []
    sim_time = 0.0

    wet = weather_summary["rain_minutes_ratio"] > 0.3

    for i, (c, L) in enumerate(zip(compounds, stint_laps), start=1):
        c = str(c).upper()
        lap_times = []

        for lap_in_stint in range(L):
            # Enhanced lap time calculation
            lap_t = (base_lap +
                    comp_delta.get(c, 0.0) +
                    degr.get(c, 0.025) * lap_in_stint)

            # Weather mismatch penalties
            is_dry_tire = c in ["SOFT", "MEDIUM", "HARD"]
            is_wet_tire = c in ["INTERMEDIATE", "WET"]

            if wet and is_dry_tire:
                lap_t += 15.0  # Severe penalty for dry tires in wet
            elif not wet and is_wet_tire:
                lap_t += 8.0  # Penalty for wet tires in dry

            # Track temperature effect on tire performance
            if weather_summary["track_temp_mean"] > 45:
                # Hot track increases degradation for softer compounds
                if c == "SOFT":
                    lap_t += 0.05 * lap_in_stint

            lap_times.append(lap_t)

        stint_sec = float(np.sum(lap_times))
        sim_time += stint_sec

        detailed.append({
            "stint": i,
            "compound": c,
            "laps": L,
            "predicted_stint_time_sec": stint_sec,
            "avg_lap_time": stint_sec / L if L > 0 else 0
        })

        # Add pit stop time (except after last stint)
        if i < len(compounds):
            sim_time += pit_loss

    # === GRU MODEL CALIBRATION ===
    gru_time_norm = float(p_time[0][0])
    gru_time_loss = np.expm1(gru_time_norm * pre["y_time_std"] + pre["y_time_mean"])

    pred_stops_class = int(np.argmax(p_stops[0]))
    pred_stops_val = int(pre["stops_le"].inverse_transform([pred_stops_class])[0])

    # Strategy deviation penalty
    stop_mismatch_penalty = abs((len(compounds) - 1) - pred_stops_val) * 5.0

    # Tire compound mismatch penalty
    pred_tire_class = int(np.argmax(p_tire[0]))
    pred_tire = pre["tire_le"].inverse_transform([pred_tire_class])[0]

    # Check if strategy includes the predicted tire
    tire_mismatch_penalty = 0.0
    if pred_tire not in compounds:
        tire_mismatch_penalty = 3.0

    # Final time: weighted blend of simulation + GRU calibration
    total = sim_time + 0.2 * gru_time_loss + stop_mismatch_penalty + tire_mismatch_penalty

    return {
        "strategy": " → ".join(compounds),
        "stops": len(compounds) - 1,
        "predicted_total_time_sec": total,
        "pit_loss_per_stop_sec": pit_loss,
        "gru_predicted_stops": pred_stops_val,
        "gru_tire_class": pred_tire,
        "gru_time_loss_sec": gru_time_loss,
        "stints": detailed,
    }

In [ ]:
print("="*70)
print("F1 STRATEGY RECOMMENDER - FASTF1 + OPENF1 COMBINED")
print("="*70)

# Create example input files if missing
if not Path(RACE_CONTEXT_FILE).exists():
    print(f"\nCreating example {RACE_CONTEXT_FILE}...")
    pd.DataFrame([{
        "team_name": "Ferrari",
        "track_name": "Imola",
        "starting_position": 4,
        "starting_compound": "MEDIUM",
        "race_laps": 63,
        "avg_tire_degradation": 0.028  # Optional: learned from FastF1
    }]).to_csv(RACE_CONTEXT_FILE, index=False)

if not Path(FORECAST_FILE).exists():
    print(f"Creating example {FORECAST_FILE}...")
    pd.DataFrame([
        {"date":"2026-05-17T13:00:00Z","air_temperature":22.1,"track_temperature":37.5,"humidity":44,"rainfall":0.0,"wind_speed":2.9},
        {"date":"2026-05-17T13:10:00Z","air_temperature":22.4,"track_temperature":38.1,"humidity":43,"rainfall":0.0,"wind_speed":3.1},
        {"date":"2026-05-17T13:20:00Z","air_temperature":22.8,"track_temperature":38.9,"humidity":41,"rainfall":0.0,"wind_speed":2.7},
    ]).to_csv(FORECAST_FILE, index=False)

# Check if model files exist
if not Path(MODEL_FILE).exists() or not Path(PREPROC_FILE).exists():
    print(f"\n⚠ Model files not found!")
    print(f"  Missing: {MODEL_FILE} and/or {PREPROC_FILE}")
    print(f"  Please run the training script first:")
    print(f"  python strategy_multitask_gru_fastf1_combined.py")
    exit(1)

# Load model and preprocessing
print(f"\nLoading model from {MODEL_FILE}...")
pre = joblib.load(PREPROC_FILE)
gru_model = keras.models.load_model(MODEL_FILE)

print(f"Loading pit loss data...")
pit_loss_map = estimate_track_pit_loss_map()

# Load race context and forecast
print(f"\nLoading race context from {RACE_CONTEXT_FILE}...")
rc = pd.read_csv(RACE_CONTEXT_FILE).iloc[0].to_dict()
fc = pd.read_csv(FORECAST_FILE)

# Parse and validate race context
rc["team_name"] = str(rc.get("team_name", "UNKNOWN_TEAM"))
rc["track_name"] = str(rc.get("track_name", "UNKNOWN_TRACK"))
rc["starting_position"] = float(rc.get("starting_position", 10))
rc["starting_compound"] = str(rc.get("starting_compound", "MEDIUM")).upper()
rc["race_laps"] = int(rc.get("race_laps", 60))
rc["avg_tire_degradation"] = float(rc.get("avg_tire_degradation", 0.025))

print("\n" + "-"*70)
print("RACE CONTEXT")
print("-"*70)
print(f"Team: {rc['team_name']}")
print(f"Track: {rc['track_name']}")
print(f"Starting Position: P{rc['starting_position']:.0f}")
print(f"Starting Tire: {rc['starting_compound']}")
print(f"Race Laps: {rc['race_laps']}")
print(f"Avg Tire Degradation: {rc['avg_tire_degradation']:.4f} s/lap")

# Process weather forecast
seq_input, ws = forecast_to_sequence_and_summary(
    fc,
    pre["weather_features"],
    pre["max_timesteps"],
    pre["seq_mean"],
    pre["seq_std"]
)

print("\n" + "-"*70)
print("WEATHER FORECAST")
print("-"*70)
print(f"Air Temperature: {ws['air_temp_mean']:.1f}°C")
print(f"Track Temperature: {ws['track_temp_mean']:.1f}°C")
print(f"Humidity: {ws['humidity_mean']:.1f}%")
print(f"Rain Probability: {ws['rain_minutes_ratio']:.1%}")
print(f"Wind Speed: {ws['wind_speed_mean']:.1f} m/s")

# Determine if wet conditions
wet = ws["rain_minutes_ratio"] > 0.2
if wet:
    print("\n⚠️  WET CONDITIONS DETECTED - Including wet tires in strategy pool")

# Generate candidate strategies
print(f"\n" + "-"*70)
print("EVALUATING STRATEGIES")
print("-"*70)
cands = build_candidates(rc["starting_compound"], max_stops=3, wet=wet)
print(f"Generated {len(cands)} candidate strategies...")

# Get pit loss for this track
pit_loss = float(pit_loss_map.get(rc["track_name"], DEFAULT_PIT_LOSS))
print(f"Estimated pit loss: {pit_loss:.1f}s per stop")

# Simulate all candidates
print("\nRunning simulations...")
results = [
    simulate_with_gru_enhanced(gru_model, pre, rc, seq_input, ws, c, pit_loss)
    for c in cands
]

# Rank results
res_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "stints"}
    for r in results
]).sort_values("predicted_total_time_sec").reset_index(drop=True)

best = float(res_df.loc[0, "predicted_total_time_sec"])
res_df["delta_to_best_sec"] = res_df["predicted_total_time_sec"] - best

# Display top 10
show = res_df.head(10).copy()
show["predicted_total_time"] = show["predicted_total_time_sec"].apply(format_time)
show["delta_to_best"] = show["delta_to_best_sec"].map(lambda x: f"+{x:.2f}s" if x > 0 else "BEST")

print("\n" + "="*70)
print("TOP 10 STRATEGIES")
print("="*70)
print(show[["strategy", "stops", "predicted_total_time", "delta_to_best"]].to_string(index=False))

# Show best strategy details
best_strategy_name = res_df.loc[0, "strategy"]
best_obj = next(r for r in results if r["strategy"] == best_strategy_name)

print("\n" + "="*70)
print("🏆 RECOMMENDED STRATEGY")
print("="*70)
print(f"Strategy: {best_obj['strategy']}")
print(f"Pit Stops: {best_obj['stops']}")
print(f"Predicted Total Time: {format_time(best_obj['predicted_total_time_sec'])}")
print(f"Assumed Pit Loss: {best_obj['pit_loss_per_stop_sec']:.1f}s per stop")
print(f"\nGRU Model Insights:")
print(f"  Preferred Stops: {best_obj['gru_predicted_stops']}")
print(f"  Predicted Tire: {best_obj['gru_tire_class']}")

print("\n" + "-"*70)
print("STINT BREAKDOWN")
print("-"*70)
for st in best_obj["stints"]:
    print(f"  Stint {st['stint']}: {st['compound']:12s} | "
          f"{st['laps']:2d} laps | "
          f"{format_time(st['predicted_stint_time_sec'])} "
          f"(avg: {st['avg_lap_time']:.2f}s)")

print("\n" + "="*70)
print("✓ Analysis complete!")
print("="*70)